# 05 — LSTM Model Training

This notebook trains the LSTM IDS model on the processed NSL-KDD sequences
and visualises training progress. Run after `02_data_preprocessing.ipynb`.

**Chapter 3, Section 3.5.3–3.5.4**

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.helpers import set_global_seed
from src.utils.paths import PROCESSED_DATA_DIR, FINAL_MODEL_DIR
from src.utils.serialization import load_processed_arrays
from src.config import get_config

cfg = get_config()
set_global_seed(cfg.seed)
print('Config loaded — window:', cfg.sequence.window_size)

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = \
    load_processed_arrays(PROCESSED_DATA_DIR)

print('X_train:', X_train.shape)
print('X_val  :', X_val.shape)
print('X_test :', X_test.shape)

In [ ]:
import json
from src.utils.constants import METADATA_JSON

with open(FINAL_MODEL_DIR / METADATA_JSON) as f:
    metadata = json.load(f)

n_classes   = metadata['n_classes']
class_names = metadata['class_names']
print('Classes:', n_classes, class_names)

In [ ]:
from src.models.model_factory import create_model
from src.models.model_utils import log_model_summary

model = create_model(
    model_type='lstm',
    input_shape=(X_train.shape[1], X_train.shape[2]),
    n_classes=n_classes,
)
log_model_summary(model)

In [ ]:
from src.training.trainer import train_lstm

result = train_lstm(
    model=model,
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    n_classes=n_classes,
    epochs=cfg.training.epochs,
    batch_size=cfg.training.batch_size,
    dataset='nsl_kdd',
)
print('Best epoch:', result.best_epoch)
print('Best val_loss:', result.best_val_loss)
print('Best val_acc:', result.best_val_accuracy)

In [ ]:
from src.visualization.training_curves import plot_training_curves

plot_training_curves(result.history, dataset='nsl_kdd')
print('Training curves saved to reports/figures/')